# Local Multi-Segment Inference

Run a local `v8-in` checkpoint on staged TIFF stacks under `data/training_surfaces/`.

Use Cell 2 to choose a trained/exported model or run directory, execute the setup cells once, optionally preload staged surfaces into RAM, then run any of the segment render cells at the end.

Each render now evaluates and visualizes the full fragment instead of cropping to the mask bounding box.

The `w062` render cell is included for convenience, but it requires `data/training_surfaces/w062/` to be staged locally before it can run.

In [2]:
MODEL_SELECTION = '/nader-fork/runs/pherc1447_plus_20260928000003/runs/ph1447_plus_20260928000003_20260930_211125_8bb87d39/checkpoints/epochepoch=5.ckpt'

In [3]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path('/nader-fork').resolve()

def _discover_model_candidates(repo: Path) -> dict[str, Path]:
    candidates = {'base': repo}
    runs_root = repo / 'runs'
    if not runs_root.is_dir():
        return candidates
    for model_path in sorted(runs_root.glob('**/model.safetensors')):
        model_dir = model_path.parent
        if not (model_dir / 'config.json').is_file():
            continue
        key = model_dir.relative_to(repo).as_posix()
        candidates[key] = model_dir
    return candidates

def _discover_run_candidates(repo: Path) -> dict[str, Path]:
    candidates = {}
    runs_root = repo / 'runs'
    if not runs_root.is_dir():
        return candidates
    for checkpoint_dir in sorted(runs_root.glob('**/checkpoints')):
        run_dir = checkpoint_dir.parent
        candidates[run_dir.name] = run_dir
        candidates[run_dir.relative_to(repo).as_posix()] = run_dir
    return candidates

def _best_checkpoint(run_dir: Path) -> Path:
    ckpt_dir = run_dir / 'checkpoints'
    best = sorted(ckpt_dir.glob('best-epoch*.ckpt'))
    if best:
        return best[-1]
    last = ckpt_dir / 'last.ckpt'
    if last.is_file():
        return last
    raise FileNotFoundError(f'no checkpoint found under {ckpt_dir}')

def _exported_model_dir_for_run(run_dir: Path) -> Path:
    return run_dir / 'exported_model'

def _exported_model_dir_for_checkpoint(checkpoint_path: Path) -> Path:
    checkpoint_path = checkpoint_path.resolve()
    return checkpoint_path.parent.parent / f'exported_{checkpoint_path.stem}'

def _export_checkpoint_to_model_dir(checkpoint: Path, model_dir: Path, *, reverse_layers: bool = False) -> Path:
    model_path = model_dir / 'model.safetensors'
    config_path = model_dir / 'config.json'
    if model_path.is_file() and config_path.is_file():
        return model_dir
    model_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        str(REPO / 'training' / 'export_checkpoint.py'),
        str(checkpoint),
        str(model_dir),
    ]
    if reverse_layers:
        cmd.append('--reverse-layers')
    subprocess.run(cmd, check=True, cwd=REPO)
    return model_dir

def _ensure_exported_run_model(run_dir: Path, *, reverse_layers: bool = False) -> Path:
    checkpoint = _best_checkpoint(run_dir)
    model_dir = _exported_model_dir_for_run(run_dir)
    return _export_checkpoint_to_model_dir(checkpoint, model_dir, reverse_layers=reverse_layers)

def _ensure_exported_checkpoint_model(checkpoint_path: Path, *, reverse_layers: bool = False) -> Path:
    if checkpoint_path.suffix != '.ckpt' or not checkpoint_path.is_file():
        raise FileNotFoundError(f'checkpoint not found: {checkpoint_path}')
    model_dir = _exported_model_dir_for_checkpoint(checkpoint_path)
    return _export_checkpoint_to_model_dir(checkpoint_path, model_dir, reverse_layers=reverse_layers)

def _resolve_model_dir(selection: str | Path) -> tuple[Path, str]:
    if isinstance(selection, Path):
        candidate = selection.expanduser()
    else:
        selection_str = str(selection)
        if selection_str in MODEL_CANDIDATES:
            return MODEL_CANDIDATES[selection_str], selection_str
        if selection_str in RUN_CANDIDATES:
            run_dir = RUN_CANDIDATES[selection_str]
            model_dir = _ensure_exported_run_model(run_dir, reverse_layers=False)
            return model_dir, selection_str
        candidate = Path(selection_str).expanduser()

    if not candidate.is_absolute():
        candidate = REPO / candidate
    candidate = candidate.resolve()

    if candidate.is_file() and candidate.suffix == '.ckpt':
        model_dir = _ensure_exported_checkpoint_model(candidate, reverse_layers=False)
        return model_dir, str(selection)

    if (candidate / 'model.safetensors').is_file() and (candidate / 'config.json').is_file():
        return candidate, str(selection)

    if (candidate / 'checkpoints').is_dir():
        model_dir = _ensure_exported_run_model(candidate, reverse_layers=False)
        return model_dir, str(selection)

    raise FileNotFoundError(
        f'could not resolve model selection {selection!r} to a model folder, checkpoint, or training run: {candidate}'
    )

MODEL_CANDIDATES = _discover_model_candidates(REPO)
RUN_CANDIDATES = _discover_run_candidates(REPO)
MODEL_DIR, MODEL_SELECTION_RESOLVED_FROM = _resolve_model_dir(MODEL_SELECTION)

SEGMENT_SPECS = {
    'pherc0211': {
        'segment_id': '20260928000003',
        'segment_name': 'PHerc0211_z7312_w080_abf',
        'title': 'PHerc0211',
    },
    # 'w060': {
    #     'segment_id': 'w060',
    #     'segment_name': 'w060',
    #     'title': 'PHerc1447 w060',
    # },
    # 'w062': {
    #     'segment_id': 'w062',
    #     'segment_name': 'w062',
    #     'title': 'PHerc1447 w062',
    # },
    # 'w058': {
    #     'segment_id': 'w058',
    #     'segment_name': 'w058',
    #     'title': 'PHerc1447 w058',
    # },
}

RUN_ID = 'v8in_base_local'
OUTPUT_DIR = REPO / 'output' / 'test_visualizations' / RUN_ID
DRY_RUN = False

print(REPO)
print(MODEL_DIR)
print(f'model selection: {MODEL_SELECTION}')
print(f'resolved from: {MODEL_SELECTION_RESOLVED_FROM}')
print('available exported model keys:')
for key in MODEL_CANDIDATES:
    print(f'  {key}')
print('available training run keys:')
for key in RUN_CANDIDATES:
    print(f'  {key}')

wrote /nader-fork/runs/pherc1447_plus_20260928000003/runs/ph1447_plus_20260928000003_20260930_211125_8bb87d39/exported_epochepoch=5/config.json and model.safetensors (340 tensors)
/nader-fork
/nader-fork/runs/pherc1447_plus_20260928000003/runs/ph1447_plus_20260928000003_20260930_211125_8bb87d39/exported_epochepoch=5
model selection: /nader-fork/runs/pherc1447_plus_20260928000003/runs/ph1447_plus_20260928000003_20260930_211125_8bb87d39/checkpoints/epochepoch=5.ckpt
resolved from: /nader-fork/runs/pherc1447_plus_20260928000003/runs/ph1447_plus_20260928000003_20260930_211125_8bb87d39/checkpoints/epochepoch=5.ckpt
available exported model keys:
  base
  runs/pherc1447_plus_20260928000003/runs/ph1447_plus_20260928000003_20260930_200322_d480ae69/exported_epochepoch=2
available training run keys:
  ph1447_plus_20260928000003_20260930_200322_d480ae69
  runs/pherc1447_plus_20260928000003/runs/ph1447_plus_20260928000003_20260930_200322_d480ae69
  ph1447_plus_20260928000003_20260930_211125_8bb87d

In [4]:
import json

MODEL_PATH = MODEL_DIR / 'model.safetensors'
CONFIG_PATH = MODEL_DIR / 'config.json'

if not MODEL_PATH.is_file() or not CONFIG_PATH.is_file():
    raise FileNotFoundError(f'missing local checkpoint under {MODEL_DIR}')

with open(CONFIG_PATH, 'r', encoding='utf-8') as handle:
    V8_CONFIG = json.load(handle)

V8_DEPTH = int(V8_CONFIG['in_depth'])
V8_TILE = int(V8_CONFIG['tile_size'])
V8_CLIP = int(V8_CONFIG['clip_max'])
EVAL_STRIDE = int(V8_CONFIG['stride'])
REVERSE_LAYERS = bool(V8_CONFIG.get('reverse_layers', False))
CPU_COUNT = max(1, os.cpu_count() or 1)
IO_THREADS = min(24, CPU_COUNT)
NUM_WORKERS = min(24, CPU_COUNT)
BATCH_SIZE = int(os.environ.get('V8_BATCH_SIZE', '64'))
MASK_CROP_MARGIN = 32
COMPOSITE_D0 = 10
COMPOSITE_D1 = 18
VOXEL_UM = 9.362

os.environ.setdefault('OMP_NUM_THREADS', str(CPU_COUNT))
os.environ.setdefault('MKL_NUM_THREADS', str(CPU_COUNT))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def surface_paths(segment_key: str) -> dict[str, Path | str]:
    if segment_key not in SEGMENT_SPECS:
        raise KeyError(f'unknown segment key: {segment_key}')
    spec = SEGMENT_SPECS[segment_key]
    surface_dir = REPO / 'data' / 'training_surfaces' / spec['segment_id']
    return {
        'segment_key': segment_key,
        'segment_id': spec['segment_id'],
        'segment_name': spec['segment_name'],
        'title': spec['title'],
        'surface_dir': surface_dir,
        'layers_dir': surface_dir / 'layers',
        'mask_path': surface_dir / 'labels' / 'mask.png',
        'label_path': surface_dir / 'labels' / 'inklabels.png',
    }

print(
    f'depth={V8_DEPTH} tile={V8_TILE} clip={V8_CLIP} stride={EVAL_STRIDE} '
    f'reverse={REVERSE_LAYERS} batch={BATCH_SIZE} workers={NUM_WORKERS} io_threads={IO_THREADS}'
)
print('surface availability:')
for key in SEGMENT_SPECS:
    paths = surface_paths(key)
    status = 'ok' if paths['surface_dir'].is_dir() else 'missing'
    print(f"  {key}: {status} -> {paths['surface_dir']}")

depth=24 tile=64 clip=200 stride=21 reverse=False batch=64 workers=24 io_threads=24
surface availability:
  pherc0211: ok -> /nader-fork/data/training_surfaces/20260928000003


In [5]:
import gc
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
from safetensors.torch import load_file as load_safetensors

try:
    from tqdm.auto import tqdm
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tqdm'], check=True)
    from tqdm.auto import tqdm

try:
    import huggingface_hub  # noqa: F401
    import safetensors  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'safetensors', 'huggingface_hub>=0.25'], check=True)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.chdir(REPO)

from ink8um import InkDetector
from ink8um.inference import coverage_mask, predict_stack, tile_positions

torch.set_num_threads(CPU_COUNT)
try:
    torch.set_num_interop_threads(min(8, CPU_COUNT))
except RuntimeError:
    pass

try:
    get_ipython().run_line_magic('matplotlib', 'agg')
except Exception:
    plt.switch_backend('agg')

Image.MAX_IMAGE_PIXELS = None
PRED_CMAP = plt.get_cmap('gray_r').copy()
PRED_CMAP.set_bad((0.78, 0.86, 0.95))
SURFACE_CACHE: dict[str, dict[str, object]] = {}
MODEL_CACHE: tuple[torch.nn.Module, str] | None = None

def load_model():
    global MODEL_CACHE
    if MODEL_CACHE is not None:
        return MODEL_CACHE
    model = InkDetector(**V8_CONFIG)
    model.load_state_dict(load_safetensors(str(MODEL_PATH)), strict=True)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    MODEL_CACHE = (model.to(device).eval(), device)
    return MODEL_CACHE

def _read_layer(path: Path) -> np.ndarray:
    layer = np.array(Image.open(path).convert('L'))
    return np.clip(layer, 0, V8_CLIP).astype(np.uint8, copy=False)

def load_stack(layers_dir: Path, *, desc: str) -> np.ndarray:
    layer_paths = [layers_dir / f'{idx:02d}.tif' for idx in range(V8_DEPTH)]
    for path in layer_paths:
        if not path.is_file():
            raise FileNotFoundError(path)

    layers: list[np.ndarray | None] = [None] * len(layer_paths)
    max_workers = min(IO_THREADS, len(layer_paths)) or 1
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {executor.submit(_read_layer, path): idx for idx, path in enumerate(layer_paths)}
        for future in tqdm(
            as_completed(future_to_idx),
            total=len(layer_paths),
            desc=f'{desc}: layers',
            unit='layer',
            leave=False,
        ):
            idx = future_to_idx[future]
            layers[idx] = future.result()

    first_shape = layers[0].shape
    for idx, layer in enumerate(layers[1:], start=1):
        if layer is None or layer.shape != first_shape:
            raise ValueError(f'layer shape mismatch at index {idx}: {getattr(layer, "shape", None)} vs {first_shape}')
    return np.stack(layers, axis=-1)

def make_composite(stack: np.ndarray) -> np.ndarray:
    return np.max(stack[..., COMPOSITE_D0:COMPOSITE_D1], axis=-1).astype(np.uint8)

def _require_surface_assets(paths: dict[str, object]) -> None:
    missing = [
        str(path)
        for path in (paths['layers_dir'], paths['mask_path'])
        if not Path(path).exists()
    ]
    if missing:
        raise FileNotFoundError(f"missing surface assets for {paths['segment_key']}: {missing}")

def preload_segment(segment_key: str) -> dict[str, object]:
    if segment_key in SURFACE_CACHE:
        return SURFACE_CACHE[segment_key]

    paths = surface_paths(segment_key)
    _require_surface_assets(paths)
    stack = load_stack(Path(paths['layers_dir']), desc=f"{paths['segment_name']} preload")
    mask = np.array(Image.open(paths['mask_path']).convert('L')) > 0
    surface = {
        **paths,
        'stack': stack,
        'mask': mask,
        'composite': make_composite(stack),
    }
    SURFACE_CACHE[segment_key] = surface
    return surface

def preload_available_segments(*segment_keys: str) -> dict[str, dict[str, object]]:
    keys = segment_keys or tuple(SEGMENT_SPECS)
    loaded = {}
    for segment_key in keys:
        paths = surface_paths(segment_key)
        if not Path(paths['surface_dir']).is_dir():
            print(f"skip {segment_key}: missing {paths['surface_dir']}")
            continue
        loaded[segment_key] = preload_segment(segment_key)
    print(f'cached segments: {sorted(loaded)}')
    return loaded

def predict_map(model, device, stack: np.ndarray, *, desc: str, reverse: bool | None = None) -> np.ndarray:
    coverage_stack = stack if not reverse else np.ascontiguousarray(stack[..., ::-1])
    points = tile_positions(coverage_mask(coverage_stack), V8_TILE, EVAL_STRIDE)
    progress_bar = tqdm(total=len(points), desc=f'{desc}: tiles', unit='tile', leave=False)

    def _progress(done: int, total: int) -> None:
        if progress_bar.total != total:
            progress_bar.total = total
        progress_bar.update(done - progress_bar.n)

    try:
        prediction = predict_stack(
            model,
            stack,
            reverse=reverse,
            stride=EVAL_STRIDE,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            device=device,
            progress=_progress,
        )
    finally:
        progress_bar.close()

    covered = np.zeros(prediction.shape, bool)
    for x, y in points:
        covered[y:y + V8_TILE, x:x + V8_TILE] = True
    prediction[~covered] = np.nan
    return prediction

def _draw_scale_bar(ax, panel_hw, px_scale):
    height, width = panel_hw
    length = 10000.0 / VOXEL_UM * px_scale
    thickness = max(2.0, min(height, width) / 250)
    pad = 0.05 * min(height, width) + thickness
    x1, y = width - pad, height - pad
    x0 = max(0.0, x1 - length)
    ax.add_patch(Rectangle((x0, y - thickness), x1 - x0, thickness, color='red'))
    ax.text(x0, y - 3 * thickness, '1 cm', color='red', fontsize=10, ha='left', va='bottom')

def _label_panel(ax, text: str, *, color: str) -> None:
    ax.text(
        0.01,
        0.99,
        text,
        transform=ax.transAxes,
        ha='left',
        va='top',
        color=color,
        fontsize=10,
        bbox={'facecolor': (0, 0, 0, 0.55), 'edgecolor': 'none', 'pad': 4},
    )

def render_segment(segment_key: str, reverse: bool = False) -> Path:
    surface = preload_segment(segment_key)
    stack = surface['stack']
    composite = surface['composite']
    effective_reverse = bool(REVERSE_LAYERS) ^ bool(reverse)
    stage_bar = tqdm(total=3, desc=f"render {surface['segment_name']}", unit='step')
    suffix = '_full_reverse.jpg' if reverse else '_full.jpg'
    out_path = OUTPUT_DIR / f"{surface['segment_name']}{suffix}"
    try:
        stage_bar.update(1)

        if DRY_RUN:
            prediction = np.full(stack.shape[:2], np.nan, np.float32)
        else:
            model, device = load_model()
            prediction = predict_map(
                model,
                device,
                stack,
                desc=surface['segment_name'],
                reverse=effective_reverse,
            )
        stage_bar.update(1)

        height, width = prediction.shape
        render_dpi = 100
        fig = plt.figure(
            figsize=(width / render_dpi, (2 * height) / render_dpi),
            dpi=render_dpi,
            facecolor='white',
        )
        pred_ax = fig.add_axes([0.0, 0.5, 1.0, 0.5])
        comp_ax = fig.add_axes([0.0, 0.0, 1.0, 0.5])

        direction_label = 'reversed depth order' if effective_reverse else 'native depth order'
        pred_ax.imshow(
            np.ma.masked_invalid(prediction),
            cmap=PRED_CMAP,
            vmin=0,
            vmax=1,
            interpolation='nearest',
        )
        _label_panel(
            pred_ax,
            f"{surface['title']} full-fragment prediction ({direction_label})",
            color='white',
        )

        comp_ax.imshow(composite, cmap='gray', vmin=0, vmax=255, interpolation='nearest')
        _label_panel(
            comp_ax,
            f"{surface['title']} full-fragment composite view ({direction_label})",
            color='white',
        )
        _draw_scale_bar(comp_ax, composite.shape, 1.0)

        for ax in (pred_ax, comp_ax):
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_axis_off()

        fig.savefig(out_path, dpi=render_dpi, facecolor='white', pil_kwargs={'quality': 92}, pad_inches=0)
        plt.close(fig)
        stage_bar.update(1)
    finally:
        stage_bar.close()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(out_path)
    return out_path

In [6]:
PRELOADED_SURFACES = preload_available_segments()
sorted(PRELOADED_SURFACES)

PHerc0211_z7312_w080_abf preload: layers:   0%|          | 0/24 [00:00<?, ?layer/s]

cached segments: ['pherc0211']


['pherc0211']

In [ ]:
render_segment('pherc0211')

render PHerc0211_z7312_w080_abf:   0%|          | 0/3 [00:00<?, ?step/s]

PHerc0211_z7312_w080_abf: tiles:   0%|          | 0/65719 [00:00<?, ?tile/s]

In [ ]:
render_segment('pherc0211', reverse=True)

In [ ]:
render_segment('w058')

In [ ]:
render_segment('w06')

In [ ]:
# holdout
render_segment('w062')